In [ ]:
"""
MEND (Model Editor Networks with Gradient Decomposition) 
Implementation on CounterFact Dataset with LLaMA 3.2-1B
 
Paper: "Fast Model Editing at Scale" (Mitchell et al., ICLR 2022)
https://arxiv.org/pdf/2110.11309
 
Dataset: CounterFact (https://rome.baulab.info/data/dsets/counterfact.json)
Base Model: meta-llama/Llama-3.2-1B
"""


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
import os
from huggingface_hub import login

login(token=secret_value_0)

In [ ]:
 
import os, json, logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Dict
import numpy as np
from dataclasses import dataclass
from tqdm import tqdm
from functools import reduce
 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
 
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
!wget https://rome.baulab.info/data/dsets/counterfact.json

# 1. DATA LOADING & DATASET

# 2. MEND NETWORK ARCHITECTURE

**Mend Wrapper around Llama**

# Training

# Eval

In [1]:
"""
MEND — Mitchell et al., ICLR 2022
Paper-faithful implementation.

KEY FACTS:
- gradient dL/dW for linear layer is rank-1: dL/dW = delta * u^T
  where u = layer INPUT, delta = grad w.r.t. layer OUTPUT
- MEND takes (u, delta) concatenated as input, NOT full gradient matrix
- At init: b1=0.1 (keeps ReLU alive), U=0 (identity transform)
- get_dW_train: z must NOT be detached — graph must flow to net.parameters()
- sign: dW = -alpha * delta_tilde^T @ u_tilde  (negative = gradient descent)
- forward_patch hook: cast back to out.dtype so downstream layers don't break
"""

import os, json, logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from dataclasses import dataclass, field
from typing import List
from tqdm import tqdm
from functools import reduce

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────
# 1. DATASET
# ──────────────────────────────────────────────

def load_counterfact(path="counterfact.json"):
    with open(path) as f:
        data = json.load(f)
    logger.info(f"Loaded {len(data)} samples")
    return data


class CFDataset(Dataset):
    def __init__(self, data, tok):
        self.data, self.tok = data, tok

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        d  = self.data[i]
        rr = d["requested_rewrite"]
        return dict(
            case_id      = d["case_id"],
            prompt       = rr["prompt"].format(rr["subject"]),
            target       = rr["target_new"]["str"],
            paraphrases  = d.get("paraphrase_prompts",  [])[:3],
            neighborhood = d.get("neighborhood_prompts",[])[:3],
        )


# ──────────────────────────────────────────────
# 2. MEND NETWORK
# ──────────────────────────────────────────────

class MENDNetwork(nn.Module):
    """
    Two residual blocks with low-rank weights.
    Input:  z = concat(u, delta)  [T, d_in+d_out]
    Output: z_tilde of same shape

    Init strategy to avoid dead ReLU:
    - V=0 (not U=0): dL/dV = U.T @ grad, dL/dU = grad @ V.T
      With V=0: U*V*z=0 but dL/dU = grad@0 = 0 → U gets no grad
      With U=0: U*V*z=0 but dL/dV = 0@grad = 0 → V gets no grad
      Solution: init U with small random, V with small random, both get grads
    - b1, b2 both set to 0.1 so ReLU inputs are positive at init
    """
    def __init__(self, d: int, rank: int = 1024):
        super().__init__()
        self.U1 = nn.Linear(rank, d,    bias=False) #low rank factorisation of MEND's weight at specific layer
        self.V1 = nn.Linear(d,    rank, bias=False)
        self.b1 = nn.Parameter(torch.full((d,), 0.1))
        self.s1 = nn.Parameter(torch.ones(d))
        self.o1 = nn.Parameter(torch.zeros(d))

        self.U2 = nn.Linear(rank, d,    bias=False)
        self.V2 = nn.Linear(d,    rank, bias=False)
        self.b2 = nn.Parameter(torch.full((d,), 0.1))
        self.s2 = nn.Parameter(torch.ones(d))
        self.o2 = nn.Parameter(torch.zeros(d))

        # small random init on both U and V so both receive gradients
        nn.init.normal_(self.U1.weight, std=0.001)
        nn.init.normal_(self.V1.weight, std=0.001)
        nn.init.normal_(self.U2.weight, std=0.001)
        nn.init.normal_(self.V2.weight, std=0.001)

        self.alpha = nn.Parameter(torch.tensor(1.0))

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        h   = z + F.relu(self.s1 * (self.U1(self.V1(z)) + self.b1) + self.o1)
        out = h + F.relu(self.s2 * (self.U2(self.V2(h)) + self.b2) + self.o2)
        return out


# ──────────────────────────────────────────────
# 3. MEND WRAPPER
# ──────────────────────────────────────────────

class MEND(nn.Module):
    LAYER = 10
    PROJ  = "up_proj"

    def __init__(self, model, tok, rank=1024, device="cuda"):
        super().__init__()
        self.model  = model
        self.tok    = tok
        self.device = device

        W = model.model.layers[self.LAYER].mlp.up_proj.weight
        self.W_name          = f"model.layers.{self.LAYER}.mlp.{self.PROJ}.weight"
        self.W               = W
        self.d_out, self.d_in = W.shape

        d = self.d_in + self.d_out
        self.net = MENDNetwork(d=d, rank=rank).float().to(device)

        logger.info(f"Target: {self.W_name}  {W.shape}")
        logger.info(f"d_in={self.d_in}  d_out={self.d_out}  rank={rank}")
        logger.info(f"||W||={W.data.float().norm():.2f}")

    def _get_module(self):
        return reduce(getattr,
                      self.W_name.replace(".weight","").split("."),
                      self.model)

    def _get_u_and_delta(self, input_ids, labels):
        """
        Forward+backward pass to capture:
          u     = layer input activations  [T, d_in]   float32
          delta = grad w.r.t. layer output [T, d_out]  float32
        Both are detached — they are inputs to the editor, not part of
        the base-model graph. The editor graph lives in net.parameters().
        """
        mod = self._get_module()
        u_list, delta_list = [], []

        def fwd_hook(m, inp, out):
            u_list.append(inp[0].detach().float().squeeze(0))

        def bwd_hook(m, grad_in, grad_out):
            if grad_out[0] is not None:
                delta_list.append(grad_out[0].detach().float().squeeze(0))

        fh = mod.register_forward_hook(fwd_hook)
        bh = mod.register_full_backward_hook(bwd_hook)

        self.model.eval()
        self.W.requires_grad_(True)
        try:
            with torch.enable_grad():
                loss = self.model(
                    input_ids=input_ids.to(self.device),
                    labels=labels.to(self.device),
                ).loss
                loss.backward()
        finally:
            self.model.zero_grad()
            self.W.requires_grad_(False)
            fh.remove()
            bh.remove()

        u     = u_list[0]     # [T, d_in]  detached float32
        delta = delta_list[0] # [T, d_out] detached float32
        return u, delta

    def _dW_from_z(self, z: torch.Tensor) -> torch.Tensor:
        """
        z: [T, d_in+d_out]
        Returns dW: [d_out, d_in], norm-capped at 2% of ||W||
        dW = -alpha * delta_tilde^T @ u_tilde
        """
        z_tilde     = self.net(z)  # z̃ = edited gradient features produced by the MEND network
        u_tilde     = z_tilde[:, :self.d_in]
        delta_tilde = z_tilde[:, self.d_in:]
        dW = -self.net.alpha * (delta_tilde.T @ u_tilde)

        # Hard norm cap: confirmed by diagnostic that 1-2% of ||W|| is the
        # sweet spot. Without this the editor produces dW/||W||=100+ which
        # destroys the model.
        w_norm  = self.W.data.float().norm()
        dw_norm = dW.detach().float().norm()
        cap     = 0.02 * w_norm
        if dw_norm > cap:
            dW = dW * (cap / (dw_norm + 1e-8))

        return dW

    def get_dW_train(self, input_ids, labels):
        """
        Training path.
        u and delta are detached from base-model graph.
        z = concat(u, delta) — NOT detached, so net.parameters() receive grads.
        Graph: z → net(z) → dW → forward_patch → L_edit → net.parameters()
        """
        u, delta = self._get_u_and_delta(input_ids, labels)
        z = torch.cat([u, delta], dim=-1)   # [T, d] — in graph via net params
        return self._dW_from_z(z)

    def get_dW_eval(self, input_ids, labels):
        """Eval path — no graph needed."""
        u, delta = self._get_u_and_delta(input_ids, labels)
        with torch.no_grad():
            z  = torch.cat([u, delta], dim=-1)
            dW = self._dW_from_z(z)
        return dW.to(self.W.dtype)

    def forward_patch(self, input_ids, dW, labels=None):
        """
        Simulate W += dW via output hook.
        Casts result back to out.dtype so downstream fp16 layers don't break.
        """
        mod = self._get_module()

        def hook(m, inp, out, _dW=dW):
            correction = inp[0].float() @ _dW.float().T
            return (out.float() + correction).to(out.dtype)

        h = mod.register_forward_hook(hook)
        self.model.eval()
        try:
            return self.model(
                input_ids=input_ids.to(self.device),
                labels=labels.to(self.device) if labels is not None else None,
            )
        finally:
            h.remove()

    def save(self):    return self.W.data.clone()
    def apply(self, dW):
        with torch.no_grad(): self.W.data += dW.detach().to(self.W.dtype)
    def restore(self, saved):
        with torch.no_grad(): self.W.data.copy_(saved)


# ──────────────────────────────────────────────
# 4. TRAINER
# ──────────────────────────────────────────────

class Trainer:
    def __init__(self, mend, lr=1e-4, c_edit=0.1, c_loc=1.0, device="cuda"):
        self.mend   = mend
        self.c_edit = c_edit
        self.c_loc  = c_loc
        self.dev    = device
        self.opt    = torch.optim.Adam(mend.net.parameters(), lr=lr)
        self.step   = 0

    def _tok(self, text):
        return self.mend.tok(
            text, return_tensors="pt",
            max_length=64, truncation=True, padding=False
        )["input_ids"]

    def _seq(self, p_ids, t_ids):
        return (
            torch.cat([p_ids, t_ids], dim=-1),
            torch.cat([torch.full_like(p_ids, -100), t_ids], dim=-1),
        )

    def train_step(self, batch):
        self.opt.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()

        p_ids = self._tok(batch["prompt"])
        t_ids = self._tok(" " + batch["target"])
        g_ids, g_lbl = self._seq(p_ids, t_ids)

        dW = self.mend.get_dW_train(g_ids, g_lbl)

        # L_edit on paraphrases (or edit prompt if no paraphrases)
        paras  = [x for x in batch["paraphrases"] if x]
        probes = paras if paras else [batch["prompt"]]
        L_edit = torch.zeros(1, device=self.dev)
        for ep in probes:
            e_ids, e_lbl = self._seq(self._tok(ep), t_ids)
            L_edit = L_edit + self.mend.forward_patch(e_ids, dW, labels=e_lbl).loss
        L_edit = L_edit / len(probes)

        # L_loc: KL(pre || post) on neighborhood prompt
        loc     = batch["neighborhood"][0] if batch["neighborhood"] else batch["prompt"]
        loc_ids = self._tok(loc).to(self.dev)
        with torch.no_grad():
            pre = self.mend.model(input_ids=loc_ids).logits.float()
        post = self.mend.forward_patch(loc_ids, dW).logits.float()
        V    = pre.size(-1)
        L_loc = F.kl_div(
            F.log_softmax(post.view(-1,V).clamp(-50,50), dim=-1),
            F.softmax(pre.view(-1,V).clamp(-50,50), dim=-1).clamp(min=1e-8),
            reduction="batchmean", log_target=False,
        )
        if not L_loc.isfinite():
            L_loc = torch.zeros(1, device=self.dev)

        L = self.c_edit * L_edit + self.c_loc * L_loc
        if L.isfinite():
            L.backward()
            nn.utils.clip_grad_norm_(self.mend.net.parameters(), 1.0)
            self.opt.step()

        self.step += 1
        dw_r = dW.detach().float().norm() / self.mend.W.data.float().norm()
        return dict(
            loss  = L.item() if L.isfinite() else 0.,
            edit  = L_edit.item(),
            loc   = L_loc.item(),
            dw_r  = dw_r.item(),
            alpha = self.mend.net.alpha.item(),
        )

    def train(self, ds, epochs=1, log_every=100):
        self.mend.model.eval()
        self.mend.net.train()
        loader = DataLoader(ds, batch_size=1, shuffle=True,
                            collate_fn=lambda x: x[0])

        for ep in range(epochs):
            tot = dict(loss=0., edit=0., loc=0., dw_r=0.)
            n   = 0
            bar = tqdm(loader, desc=f"Epoch {ep+1}/{epochs}")
            for batch in bar:
                try:
                    m = self.train_step(batch)
                    self.mend.model.eval()
                    if np.isfinite(m["loss"]) and m["loss"] > 0:
                        for k in tot: tot[k] += m[k]
                        n += 1
                    if n:
                        bar.set_postfix(
                            loss  = f"{tot['loss']/n:.3f}",
                            edit  = f"{tot['edit']/n:.3f}",
                            loc   = f"{tot['loc']/n:.4f}",
                            dw    = f"{tot['dw_r']/n:.4f}",
                            alpha = f"{m['alpha']:.3f}",
                        )
                    if self.step % log_every == 0 and n:
                        logger.info(
                            f"step={self.step}"
                            f"  loss={tot['loss']/n:.4f}"
                            f"  edit={tot['edit']/n:.4f}"
                            f"  loc={tot['loc']/n:.5f}"
                            f"  dw={tot['dw_r']/n:.4f}"
                            f"  alpha={m['alpha']:.4f}"
                        )
                except Exception as e:
                    logger.warning(f"step error: {e}")
                    self.mend.model.eval()
                    torch.cuda.empty_cache()

            logger.info(
                f"Epoch {ep+1}:"
                f"  loss={tot['loss']/max(n,1):.4f}"
                f"  edit={tot['edit']/max(n,1):.4f}"
                f"  loc={tot['loc']/max(n,1):.5f}"
                f"  dw={tot['dw_r']/max(n,1):.4f}"
                f"  alpha={self.mend.net.alpha.item():.4f}"
                f"  ({n}/{len(loader)})"
            )


# ──────────────────────────────────────────────
# 5. EVALUATION
# ──────────────────────────────────────────────

def gen(model, tok, prompt, max_new=15, device="cuda"):
    model.eval()
    inp = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=max_new,
            do_sample=False, pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inp["input_ids"].shape[-1]:],
                      skip_special_tokens=True).strip()


@dataclass
class Result:
    case_id:  int
    prompt:   str
    target:   str
    pred:     str
    es:       bool
    ps:       bool
    ls:       bool
    para_res: List[bool] = field(default_factory=list)
    loc_res:  List[bool] = field(default_factory=list)


def eval_one(mend, sample, device="cuda", verbose=False):
    rr  = sample["requested_rewrite"]
    ep  = rr["prompt"].format(rr["subject"])
    nt  = rr["target_new"]["str"]
    tok = mend.tok

    xe     = tok(ep,     return_tensors="pt")["input_ids"].to(device)
    ye     = tok(" "+nt, return_tensors="pt")["input_ids"].to(device)
    ids    = torch.cat([xe, ye], dim=-1)
    labels = torch.cat([torch.full_like(xe, -100), ye], dim=-1)

    nbrs    = sample.get("neighborhood_prompts", [])[:3]
    pre_nbr = {p: gen(mend.model, tok, p, device=device) for p in nbrs}

    dW = mend.get_dW_eval(ids, labels)

    if verbose:
        r = dW.float().norm() / mend.W.data.float().norm()
        print(f"  dW/||W||={r:.4f}  alpha={mend.net.alpha.item():.4f}")

    saved = mend.save()
    try:
        mend.apply(dW)
        mend.model.eval()
        pred     = gen(mend.model, tok, ep, device=device)
        es       = nt.lower() in pred.lower()
        para_res = [
            nt.lower() in gen(mend.model, tok, pp, device=device).lower()
            for pp in sample.get("paraphrase_prompts", [])[:3]
        ]
        ps = all(para_res) if para_res else False
        p5 = lambda t: " ".join(t.strip().split()[:5])
        loc_res = [
            p5(gen(mend.model, tok, p, device=device)) == p5(pre)
            for p, pre in pre_nbr.items()
        ]
        ls = all(loc_res) if loc_res else True
    finally:
        mend.restore(saved)
        mend.model.eval()

    return Result(case_id=sample["case_id"], prompt=ep, target=nt, pred=pred,
                  es=es, ps=ps, ls=ls, para_res=para_res, loc_res=loc_res)


def eval_batch(mend, samples, device, n=50, verbose=False):
    mend.model.eval()
    results = []
    for s in tqdm(samples[:n], desc="Evaluating"):
        try:
            r = eval_one(mend, s, device=device, verbose=verbose)
            results.append(r)
            if verbose:
                print(f"  [{r.case_id}] {r.prompt!r} -> {r.target!r}")
                print(f"  pred={r.pred!r}  ES={r.es} PS={r.ps} LS={r.ls}")
        except Exception as e:
            logger.warning(f"eval error: {e}")
    if not results: return {}
    es = np.mean([r.es for r in results])
    ps = np.mean([r.ps for r in results])
    ls = np.mean([r.ls for r in results])
    logger.info(
        f"\n{'='*40}\n"
        f"N={len(results)}\n"
        f"ES={es*100:.1f}%  PS={ps*100:.1f}%  LS={ls*100:.1f}%\n"
        f"{'='*40}"
    )
    return dict(ES=es, PS=ps, LS=ls)


# ──────────────────────────────────────────────
# 6. SANITY CHECK
# ──────────────────────────────────────────────

def sanity(mend, device="cuda", label=""):
    print(f"\n=== SANITY CHECK {label} ===")
    with open("counterfact.json") as f:
        s = json.load(f)[0]["requested_rewrite"]
    tok    = mend.tok
    prompt = s["prompt"].format(s["subject"])
    nt     = s["target_new"]["str"]
    nid    = tok(" "+nt, add_special_tokens=False)["input_ids"][0]
    xe     = tok(prompt, return_tensors="pt")["input_ids"].to(device)
    ye     = tok(" "+nt, return_tensors="pt")["input_ids"].to(device)
    ids    = torch.cat([xe, ye], dim=-1)
    labels = torch.cat([torch.full_like(xe,-100), ye], dim=-1)

    def rank():
        with torch.no_grad():
            lg = mend.model(input_ids=xe).logits[0,-1]
        return (lg.argsort(descending=True) == nid).nonzero().item() + 1

    r0  = rank()
    pre = gen(mend.model, tok, prompt, device=device)
    dW  = mend.get_dW_eval(ids, labels)
    ratio = dW.float().norm() / mend.W.data.float().norm()
    print(f"  prompt   : {prompt}")
    print(f"  target   : {nt}")
    print(f"  pre-edit : {pre!r}  rank={r0}")
    print(f"  dW/||W|| : {ratio:.5f}  alpha={mend.net.alpha.item():.4f}")

    saved = mend.save()
    try:
        mend.apply(dW)
        mend.model.eval()
        r1   = rank()
        post = gen(mend.model, tok, prompt, device=device)
    finally:
        mend.restore(saved)
        mend.model.eval()

    print(f"  post-edit: {post!r}  rank={r1}")
    print(f"  rank     : {r0} → {r1}  ({'✓ improved' if r1 < r0 else '✗'})")
    print(f"=== END ===\n")


# ──────────────────────────────────────────────
# 7. GRAD CHECK
# ──────────────────────────────────────────────

def check_grads(mend, device="cuda"):
    print("\n=== GRAD CHECK ===")
    with open("counterfact.json") as f:
        s = json.load(f)[0]["requested_rewrite"]
    tok    = mend.tok
    xe     = tok(s["prompt"].format(s["subject"]),
                 return_tensors="pt")["input_ids"].to(device)
    ye     = tok(" "+s["target_new"]["str"],
                 return_tensors="pt")["input_ids"].to(device)
    ids    = torch.cat([xe, ye], dim=-1)
    labels = torch.cat([torch.full_like(xe,-100), ye], dim=-1)

    mend.net.train()
    mend.net.zero_grad()
    dW = mend.get_dW_train(ids, labels)
    mend.forward_patch(ids, dW, labels=labels).loss.backward()

    ok = True
    for name, param in mend.net.named_parameters():
        g    = param.grad
        norm = g.norm().item() if g is not None else 0.
        status = "✓" if norm > 1e-12 else "✗ (zero)"
        print(f"  {name:20s}: {norm:.3e}  {status}")
        if norm <= 1e-12:
            ok = False
    mend.net.zero_grad()
    print(f"  → {'ALL OK ✓' if ok else 'BROKEN ✗'}\n")
    return ok


# ──────────────────────────────────────────────
# 8. MAIN
# ──────────────────────────────────────────────

def main():
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
    MODEL_NAME = "meta-llama/Llama-3.2-1B"
    CF_PATH    = "counterfact.json"
    SAVE_PATH  = "mend_editor.pt"
    N_TRAIN    = 1000
    N_EVAL     = 100
    EPOCHS     = 5
    LR         = 1e-4
    RANK       = 1024
    C_EDIT     = 0.1
    C_LOC      = 1.0

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map=DEVICE)
    model.eval()

    raw      = load_counterfact(CF_PATH)
    train_ds = CFDataset(raw[:N_TRAIN], tok)
    eval_ss  = raw[:N_EVAL]

    mend = MEND(model, tok, rank=RANK, device=DEVICE)

    if not check_grads(mend, DEVICE):
        logger.error("Gradient check failed — fix before training.")
        return

    sanity(mend, DEVICE, label="BEFORE TRAINING")

    trainer = Trainer(mend, lr=LR, c_edit=C_EDIT, c_loc=C_LOC, device=DEVICE)
    trainer.train(train_ds, epochs=EPOCHS, log_every=100)

    torch.save(mend.net.state_dict(), SAVE_PATH)
    logger.info(f"Saved to {SAVE_PATH}")

    sanity(mend, DEVICE, label="AFTER TRAINING")

    logger.info("=== Eval on first N_EVAL training samples ===")
    eval_batch(mend, eval_ss, DEVICE, n=N_EVAL, verbose=False)


if __name__ == "__main__":
    main()

2026-03-14 14:46:01,768 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-14 14:46:01,838 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-14 14:46:01,907 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-14 14:46:01,975 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-1B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-14 14:46:02,048 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-1B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-14 14:46:03,386 INFO HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-1B "HTTP/1.1 200 OK"
2026-03-14 14:46:03,511 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

2026-03-14 14:46:06,056 INFO HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-03-14 14:46:06,384 INFO Loaded 21919 samples
2026-03-14 14:46:06,938 INFO Target: model.layers.10.mlp.up_proj.weight  torch.Size([8192, 2048])
2026-03-14 14:46:06,939 INFO d_in=2048  d_out=8192  rank=1024
2026-03-14 14:46:06,961 INFO ||W||=70.46



=== GRAD CHECK ===
  b1                  : 4.376e-01  ✓
  s1                  : 4.376e-02  ✓
  o1                  : 4.376e-01  ✓
  b2                  : 4.376e-01  ✓
  s2                  : 4.376e-02  ✓
  o2                  : 4.376e-01  ✓
  alpha               : 9.971e-02  ✓
  U1.weight           : 1.437e-01  ✓
  V1.weight           : 1.384e-01  ✓
  U2.weight           : 1.994e-01  ✓
  V2.weight           : 2.041e-01  ✓
  → ALL OK ✓


=== SANITY CHECK BEFORE TRAINING ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

Epoch 1/5: 100%|██████████| 1000/1000 [08:06<00:00,  2.06it/s, alpha=1.095, dw=0.0200, edit=17.198, loc=0.0113, loss=1.731]
2026-03-14 14:54:15,342 INFO Epoch 1:  loss=1.7311  edit=17.1978  loc=0.01133  dw=0.0200  alpha=1.0954  (1000/1000)
Epoch 2/5: 100%|██████████| 1000/1000 [08:04<00:00,  2.06it/s, alpha=1.185, dw=0.0200, edit=17.105, loc=0.0122, loss=1.723]
2026-03-14 15:02:20,159 INFO Epoch 2:  loss=1.7227  edit=17.1053  loc=0.01216  dw=0.0200  alpha=1.1848  (1000/1000)
Epoch 3/5: 100%|██████████| 1000/1000 [08:04<00:00,  2.06it/s, alpha=1.221, dw=0.0081, edit=18.305, loc=0.0058, loss=1.836]
2026-03-14 15:10:24,991 INFO Epoch 3:  loss=1.8362  edit=18.3048  loc=0.00575  dw=0.0081  alpha=1.2208  (1000/1000)
Epoch 4/5: 100%|██████████| 1000/1000 [08:04<00:00,  2.06it/s, alpha=1.224, dw=0.0005, edit=19.039, loc=0.0018, loss=1.906]
2026-03-14 15:18:29,932 INFO Epoch 4:  loss=1.9057  edit=19.0388  loc=0.00181  dw=0.0005  alpha=1.2238  (1000/1000)
Epoch 5/5: 100%|██████████| 1000/1000 [0


=== SANITY CHECK AFTER TRAINING ===
  prompt   : The mother tongue of Danielle Darrieux is
  target   : English
  pre-edit : 'French. She was born in Paris on 15th December 1903'  rank=6
  dW/||W|| : 0.00000  alpha=1.2257


2026-03-14 15:26:38,582 INFO === Eval on first N_EVAL training samples ===


  post-edit: 'French. She was born in Paris on 15th December 1903'  rank=6
  rank     : 6 → 6  (✗)
=== END ===



Evaluating: 100%|██████████| 100/100 [05:19<00:00,  3.20s/it]
2026-03-14 15:31:58,318 INFO 
N=100
ES=0.0%  PS=0.0%  LS=100.0%
